# Notebook 3 - Heartbeats, Truncation, and Real-World HWMs

The first two notebooks introduced the high-water mark. This one fills in the pieces a real system needs:

1. **How followers learn the HWM** - via heartbeats from the leader.
2. **Truncation** - what a follower does when it reconnects with extra entries beyond the new leader's HWM.
3. **Reading from followers safely** - the same rule (`offset <= HWM`) protects follower reads too.
4. **Where you meet HWMs in production** - Kafka, Raft, MongoDB, PostgreSQL, ZooKeeper.

We will build a slightly richer simulation that exchanges *messages* between nodes instead of mutating shared state directly.


## Setup

```bash
cd 02-distributed-primitives/high-water-mark
uv sync
```

Pick the `.venv` kernel in VS Code, reload the window if needed.


## A tiny replicated log with heartbeats

We model three message types - just enough to see HWM propagation:

- `AppendEntries(offset, entry, leader_hwm)` - leader to follower, ships an entry **and** the leader's current HWM.
- `Ack(follower, offset)` - follower to leader, "I durably stored up to this offset".
- `Heartbeat(leader_hwm)` - leader to follower, periodic update of the HWM (used when there are no new entries).

Each follower keeps its own `commit_index` (its local view of the HWM) and only exposes `log[: commit_index + 1]` to clients.


In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional

@dataclass
class Follower:
    name: str
    log: List[str] = field(default_factory=list)
    commit_index: int = -1  # follower's known HWM

    def receive_append(self, offset: int, entry: str, leader_hwm: int) -> Optional[int]:
        # Only accept the entry if it lines up with our log.
        if offset == len(self.log):
            self.log.append(entry)
        elif offset < len(self.log):
            # Already have something at this offset; in real Raft we would compare terms.
            self.log[offset] = entry
            del self.log[offset + 1:]  # truncate divergent suffix
        else:
            # Gap - refuse, leader will re-send earlier entries.
            return None
        # Followers can advance their commit_index up to what the leader says,
        # but never past their own log length.
        self.commit_index = min(leader_hwm, len(self.log) - 1)
        return offset  # ack offset

    def heartbeat(self, leader_hwm: int) -> None:
        self.commit_index = min(leader_hwm, len(self.log) - 1)

    def visible_to_clients(self) -> List[str]:
        return self.log[: self.commit_index + 1]


@dataclass
class Leader:
    name: str = "L"
    log: List[str] = field(default_factory=list)
    match_index: Dict[str, int] = field(default_factory=dict)
    quorum: int = 2

    def append(self, entry: str) -> int:
        self.log.append(entry)
        return len(self.log) - 1

    @property
    def high_water_mark(self) -> int:
        leader_offset = len(self.log) - 1
        offsets = sorted([leader_offset, *self.match_index.values()], reverse=True)
        # If we do not yet have enough nodes (leader + acks) to form a quorum,
        # nothing is committed yet.
        if len(offsets) < self.quorum:
            return -1
        return offsets[self.quorum - 1]

    def replicate_to(self, follower: Follower, offset: int) -> None:
        ack = follower.receive_append(offset, self.log[offset], self.high_water_mark)
        if ack is not None:
            self.match_index[follower.name] = max(self.match_index.get(follower.name, -1), ack)

    def heartbeat_to(self, follower: Follower) -> None:
        follower.heartbeat(self.high_water_mark)


## Step through a normal cycle

Watch how the HWM propagates: it is *one round-trip behind* the data. After the leader replicates entry `N`, a follower only learns that `N` is committed on the **next** message - either the next `AppendEntries` or a `Heartbeat`.


In [ ]:
leader = Leader()
f1, f2 = Follower("f1"), Follower("f2")

def step(label, fn):
    fn()
    print(
        f"{label:<28}  hwm={leader.high_water_mark}  "
        f"f1.commit={f1.commit_index} f1.visible={f1.visible_to_clients()}  "
        f"f2.commit={f2.commit_index} f2.visible={f2.visible_to_clients()}"
    )

step("append A",        lambda: leader.append("A"))
step("ship A -> f1",    lambda: leader.replicate_to(f1, 0))
step("ship A -> f2",    lambda: leader.replicate_to(f2, 0))  # quorum reached, hwm=0
step("heartbeat -> f1", lambda: leader.heartbeat_to(f1))     # f1 learns hwm=0
step("heartbeat -> f2", lambda: leader.heartbeat_to(f2))     # f2 learns hwm=0

step("append B",        lambda: leader.append("B"))
step("ship B -> f1",    lambda: leader.replicate_to(f1, 1))  # leader+f1 = quorum, hwm=1
# Note: f1 learned hwm=1 piggybacked on the AppendEntries above.
step("heartbeat -> f2", lambda: leader.heartbeat_to(f2))     # f2 still missing B, cannot expose it


Two important things to notice:

1. **`f2.visible` never shows `B`** even though the cluster has committed it - `f2` can only expose what it physically has. That is correct.
2. The HWM update for `f1` arrived **piggybacked** on the next `AppendEntries`. In practice, systems use heartbeats to deliver HWM updates whenever there is no new data flowing.

## Truncation after a leadership change

Suppose the leader appended an entry `X` that no follower received, then crashed. `f1` becomes the new leader. When the old leader recovers and rejoins as a follower, it has an extra entry beyond the cluster's HWM. The new leader will tell it to **truncate** that suffix.


In [ ]:
# Leader appended X but never replicated it.
leader.append("X")
print("old leader log:", leader.log,
      "committed:", leader.log[: leader.high_water_mark + 1])

# Crash leader. f1 (which has [A, B]) becomes the new leader.
new_leader = Leader(name="L2", log=list(f1.log))
# Old leader rejoins as a follower; its log has the bogus 'X'.
old_leader_as_follower = Follower("old", log=list(leader.log))
print("rejoining follower log (before truncate):", old_leader_as_follower.log)

# New leader sees the rejoiner has length 3 but the new leader only has 2 entries.
# It re-sends what it has from offset 0 onward; receive_append truncates the divergent tail.
for off in range(len(new_leader.log)):
    new_leader.replicate_to(old_leader_as_follower, off)

print("rejoining follower log (after  truncate):", old_leader_as_follower.log)


`X` is gone - and that is the **right** outcome, because it was never committed and no client was ever told it succeeded.

## Where this shows up in real systems

| System | Name they use | Notes |
|---|---|---|
| **Apache Kafka** | *High Watermark* | Per-partition; equals the smallest log-end-offset of all in-sync replicas (ISR). Consumers can only read up to it. |
| **Raft** (etcd, Consul, CockroachDB, TiKV) | `commitIndex` | Leader advances `commitIndex` once a log entry is on a majority; followers learn the new value via `AppendEntries` (which doubles as the heartbeat). |
| **MongoDB** | *majority commit point* | Tracked across replica set members; `readConcern: "majority"` exposes only data at-or-below it. |
| **PostgreSQL streaming replication** | `flush_lsn` / `apply_lsn` | Synchronous replicas only ack a transaction once it is flushed; clients with `synchronous_commit=on` wait for that ack - the same idea, expressed in WAL byte offsets. |
| **ZooKeeper / ZAB** | *committed zxid* | Leader proposes, a quorum of followers ack, then leader broadcasts COMMIT. |

The vocabulary differs but the pattern is the same: **do not expose data until a quorum of nodes durably has it**.

## Final takeaways

- The HWM is the boundary between *durable & visible* and *tentative & hidden*.
- It moves forward only as quorum acks arrive.
- Followers learn it from the leader (via heartbeats / piggybacked metadata) and may need to **truncate** when they have entries past the HWM after a leader change.
- Whenever you see "majority writes", "ISR", `commitIndex`, or "majority read concern", a high-water mark is doing the work behind the scenes.

## Try it yourself

- Change `quorum` to `3` in `Leader` and add a third follower. How does the HWM change?
- Make `replicate_to` randomly drop messages and add a retry loop. The HWM should still be safe - it just advances slower.
- Add a `read(offset)` method to `Follower` that **refuses** reads above `commit_index`. That is the consumer-side guardrail that makes follower reads safe.
